In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import statistics
import math
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import pandas as pd

KeyboardInterrupt: 

In [ ]:
xrdsNorESMall_levs = xr.open_dataset("/share/sabl0586/all_stations_NorESM_OsloAero_prcp2szdst_f19_f19_noresmv211_corr_ilevall_levs_4Peter.nc")
SMRNorlevs =  xrdsNorESMall_levs.sel(station='SMR-II')


In [ ]:
def NorExtract(Data):
    ds=xr.Dataset()
    for i in range(1,16):
        if f'SIGMA{i:02d}' not in Data:
            continue   
        a = Data[f'SIGMA{i:02d}']
        ds[f'SIGMA{i:02d}'] = a
        b = Data[f'NMR{i:02d}']
        ds[f'NMR{i:02d}'] = b
        c = Data[f'NCONC{i:02d}']
        ds[f'NCONC{i:02d}']=c
    ds['FREQL'] = Data['FREQL']
    ds['AWNC'] = Data['AWNC']
    ds['WSUB'] = Data['WSUB'] 
    ds['T'] = Data['T']
    return ds
        

In [ ]:
ls_oa = ['SOA_NA', 'SOA_A1', 'OM_AC', 'OM_AI', 'OM_NI','SOA_NA_OCW', 'SOA_A1_OCW', 'OM_AC_OCW', 'OM_AI_OCW', 'OM_NI_OCW']
ls_so4 = ['SO4_NA', 'SO4_A1','SO4_A2','SO4_AC', 'SO4_PR','SO4_NA_OCW', 'SO4_A1_OCW','SO4_A2_OCW','SO4_AC_OCW', 'SO4_PR_OCW',]
ls_seasalt = ['SS_A1', 'SS_A2','SS_A1_OCW', 'SS_A2_OCW',]
ls_dust = ['DST_A2','DST_A2_OCW',]
Ls_bc = ['BC_N','BC_AX','BC_NI','BC_A','BC_AI','BC_AC','BC_N_OCW','BC_AX_OCW','BC_NI_OCW','BC_A_OCW','BC_AI_OCW','BC_AC_OCW',]

In [ ]:
Output=xr.Dataset()
Output = NorExtract(SMRNorlevs)

In [ ]:
#!rm /share/pech2273/SMRPeterVariables2.nc

Output.to_netcdf('/share/pech2273/SMRPeterVariablesWithMode14.nc')


In [ ]:
dsECEarthallevs=xr.open_dataset("/share/sabl0586/all_stations_EC-Earth_PRCP2SZDST_ilevall_levs_4Peter.nc")
SMRECEarth = dsECEarthallevs.sel(station = 'SMR-II')
SMRECEarth

In [ ]:
diam_variables=['RDRY_NUS','RDRY_AIS','RDRY_ACS','RWET_AII','RDRY_COS','RWET_ACI','RWET_COI',]
Numb_variables = ['N_NUS','N_AIS','N_ACS','N_AII','N_COS','N_ACI','N_COI',]
ifs_vars = ['var130','var54','var131','var132','var20','var21','var22','var248']
ifs_vars_names = ['temp','pres','U','V','CDNC','re_liq','Liquid_Cloud_time','Cloud_Frac']

In [ ]:
ds_ifs=xr.Dataset()

SMRECEarth['lev']=SMRECEarth['pressure'].mean('time')
for ifs in ifs_vars:
    ds_ifs[ifs] = SMRECEarth[ifs]
ds_ifs = ds_ifs[['var130','var54','var131','var132','var132','var20','var21','var22','var248']].isel(lev=0).drop_vars('lev')
ds_ifs['lev_ifs'] = ds_ifs['var54'].mean('time')
for ifs in ifs_vars:
    ds_ifs[ifs] = ds_ifs.sel(lev_ifs = SMRECEarth['lev'], method='nearest')[ifs]

    
ds_ifs = ds_ifs.drop_vars('lev_ifs')
ds_ifs['lev'] = ds_ifs['lev']/100
    

In [ ]:
ds_ifs 

In [ ]:
ds_ifs['var130'].dropna('time').mean('time').plot(y = 'lev', yscale = 'log', ylim = [1000,.1], label= 'EC Earth')
Output['T'].mean('time').plot(y = 'lev', label = 'Nor ESM')
plt.legend()

In [ ]:
z = SMRNorlevs['lev'].to_numpy()
z = np.flip(z)
plt.plot(z, label = 'NorESM')
plt.plot(SMRECEarth['lev']/100, label = 'ECEarth')
plt.legend()

In [ ]:
def ECextract (Data):
    ds=xr.Dataset()
    for i in range(0,len(diam_variables)):
        a = Data[f'{diam_variables[i]}']
        ds[f'{diam_variables[i]}'] = a
        b = Data[f'{Numb_variables[i]}']
        ds[f'{Numb_variables[i]}'] = b
    ds['lev'] = ds['lev']/100    
    #ds['CDNC'] = Data['var20']
    #ds = ds.drop_vars('lev_ifs')
    #ds['TotalCloudCover'] = Data['var164']
    #ds['CloudCoverFrac'] = Data['var248']
    #ds['CloudTime'] = Data['var22']
    #ds['pressure'] = Data['pressure']
    return ds

In [ ]:
OutputEC = ECextract(SMRECEarth)

In [ ]:
OutputEC['lev']

!rm /share/pech2273/SMRECEARTHPeterVariables2.nc

In [ ]:
!rm /share/pech2273/SMRECEARTHPeterVariables.nc
!rm /share/pech2273/SMRECEARTHPeterVariablesIFS.nc
OutputEC.to_netcdf('/share/pech2273/SMRECEARTHPeterVariables.nc')
ds_ifs.to_netcdf('/share/pech2273/SMRECEARTHPeterVariablesIFS.nc')

In [ ]:
NORls_oa = ['SOA_NA', 'SOA_A1', 'OM_AC', 'OM_AI', 'OM_NI','SOA_NA_OCW', 'SOA_A1_OCW', 'OM_AC_OCW', 'OM_AI_OCW', 'OM_NI_OCW']
NORls_so4 = ['SO4_NA', 'SO4_A1','SO4_A2','SO4_AC', 'SO4_PR','SO4_NA_OCW', 'SO4_A1_OCW','SO4_A2_OCW','SO4_AC_OCW', 'SO4_PR_OCW',]
NORls_seasalt = ['SS_A1', 'SS_A2','SS_A1_OCW', 'SS_A2_OCW',]
NORls_dust = ['DST_A2','DST_A2_OCW',]
NORls_bc = ['BC_N','BC_AX','BC_NI','BC_A','BC_AI','BC_AC','BC_N_OCW','BC_NI_OCW','BC_A_OCW','BC_AI_OCW','BC_AC_OCW',]#'BC_AX_OCW',
ECls_oa = ['M_SOANUS','M_POMAIS','M_SOAAIS','M_POMACS','M_SOAACS','M_POMAII', 'M_SOAAII',]
ECls_so4 = ['M_SO4NUS','M_SO4ACS',] #'M_SO4AIS'
ECls_seasalt= ['M_SSACS'] 
ECls_dust = ['M_DUACI','M_DUACS'] 
ECls_bc = ['M_BCACS','M_BCAII','M_BCAIS',] 

In [ ]:
M_SO4AIS_es =(SMRECEarth['M_SO4ACS']/(SMRECEarth['M_BCACS']+SMRECEarth['M_POMACS']+SMRECEarth['M_SOAACS']))*(SMRECEarth['M_BCAIS']+SMRECEarth['M_POMAIS']+SMRECEarth['M_SOAAIS'])

In [ ]:
M_SO4AIS_es

In [ ]:
NorBCMass = sum(SMRNorlevs[i] for i in NORls_bc)
NorOAMass = sum(SMRNorlevs[i] for i in NORls_oa)
NorSO4Mass = sum(SMRNorlevs[i] for i in NORls_so4)
NorSeasaltMass = sum(SMRNorlevs[i] for i in NORls_seasalt)
NorDustMass = sum(SMRNorlevs[i] for i in NORls_dust)
Norls_Mass = [NorBCMass,  NorOAMass, NorSO4Mass, NorSeasaltMass, NorDustMass] 


In [ ]:
ECBCMass = sum(SMRECEarth[i] for i in ECls_bc)
ECOAMass = sum(SMRECEarth[i] for i in ECls_oa)
ECSO4Mass = sum(SMRECEarth[i] for i in ECls_so4)+M_SO4AIS_es
ECSeasaltMass = sum(SMRECEarth[i] for i in ECls_seasalt)
ECDustMass = sum(SMRECEarth[i] for i in ECls_dust)
ECls_Mass = [ECBCMass,  ECOAMass, ECSO4Mass, ECSeasaltMass, ECDustMass] 
#MassSOCAI = MassSO4ACS*RatioVolume(AIK/ACC)

In [ ]:
NorBCMassFrac = NorBCMass/sum( i for i in Norls_Mass)
NorOAMassFrac = NorOAMass/sum( i for i in Norls_Mass)
NorSO4MassFrac = NorSO4Mass/sum( i for i in Norls_Mass)
NorSeasaltMassFrac = NorSeasaltMass/sum( i for i in Norls_Mass)
NorDustMassFrac = NorDustMass/sum( i for i in Norls_Mass)


In [ ]:
ECBCMassFrac = ECBCMass/sum( i for i in ECls_Mass)
ECOAMassFrac = ECOAMass/sum( i for i in ECls_Mass)
ECSO4MassFrac = ECSO4Mass/sum( i for i in ECls_Mass)
ECSeasaltMassFrac = ECSeasaltMass/sum( i for i in ECls_Mass)
ECDustMassFrac = ECDustMass/sum( i for i in ECls_Mass)

In [ ]:

plt.subplot(1,2,1)
plt.pie(x = [ECBCMassFrac.isel(lev = 0).mean('time'), ECOAMassFrac.isel(lev = 0).mean('time'),\
             ECSO4MassFrac.isel(lev = 0).mean('time'), ECSeasaltMassFrac.isel(lev = 0).mean('time'),\
             ECDustMassFrac.isel(lev = 0).mean('time')],colors=['black', 'green','red','blue','purple'], labels = ['BC','OA','AS','Seasalt', 'Dust'])
plt.title('Average ECEarth Mass Frac')
plt.subplot(1,2,2)
plt.pie(x = [NorBCMassFrac.isel(lev = -1).mean('time'), NorOAMassFrac.isel(lev = -1).mean('time'),
             NorSO4MassFrac.isel(lev = -1).mean('time'), NorSeasaltMassFrac.isel(lev = -1).mean('time'),\
             NorDustMassFrac.isel(lev = -1).mean('time')], colors=['black', 'green','red','blue','purple'], labels = ['BC','OA','AS','Seasalt', 'Dust'])
plt.title('Average NorESM Mass Frac')
plt.tight_layout()
plt.show()

In [ ]:
ECMassFrac_ds = xr.Dataset({'BCMassFrac' : ECBCMassFrac, 'OAMassFrac' :ECOAMassFrac,'SO4MassFrac' : ECSO4MassFrac, 'SeasaltMassFrac' :ECSeasaltMassFrac, 'DustMassFrac' :ECDustMassFrac,\
                            'BCMass' : ECBCMass, 'OAMass' :ECOAMass,'SO4Mass' : ECSO4Mass, 'SeasaltMass' :ECSeasaltMass, 'DustMass' :ECDustMass})

In [ ]:
NorMassFrac_ds = xr.Dataset({'BCMassFrac' : NorBCMassFrac, 'OAMassFrac' :NorOAMassFrac,'SO4MassFrac' : NorSO4MassFrac, 'SeasaltMassFrac' :NorSeasaltMassFrac, 'DustMassFrac' :NorDustMassFrac,\
                            'BCMass' : NorBCMass, 'OAMass' :NorOAMass,'SO4Mass' : NorSO4Mass, 'SeasaltMass' :NorSeasaltMass, 'DustMass' :NorDustMass}) 

In [ ]:
!rm /share/pech2273/ECMassFrac.nc
!rm /share/pech2273/NorMassFrac.nc
ECMassFrac_ds.to_netcdf('/share/pech2273/ECMassFrac.nc')
NorMassFrac_ds.to_netcdf('/share/pech2273/NorMassFrac.nc')

In [ ]:
Obs_ds = xr.open_dataset("/share/nibe4885/obs_SMEAR_II_Niels.nc")